In [ ]:
# -*- coding: utf-8 -*-
"""
30-Community Bounded Sensitive Retrieval Pilot – RESUMABLE + VERBOSE
Adds immediate console output and per-community progress.
"""

import re
import time
from pathlib import Path

import pandas as pd
import requests

API = "https://arctic-shift.photon-reddit.com/api/posts/search"
START = "2017-01-01"
BEFORE = "2026-08-31"
TIMEOUT = 25
SLEEP = 0.05
KNOWN_ID = "1v37oob"

OUTDIR = Path("bounded_sensitive_pilot_30_resumable_verbose")
OUTDIR.mkdir(exist_ok=True)

SUMMARY_CKPT = OUTDIR / "checkpoint_summary.csv"
POSTS_CKPT = OUTDIR / "checkpoint_posts.csv"
AUDIT_CKPT = OUTDIR / "checkpoint_audit.csv"

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Masterarbeit-Bounded-Sensitive-Retrieval-Pilot/1.2 academic research"
})

BROAD_AI = [
    "ai", "artificial intelligence", "generative ai", "genai",
    "generated", "synthetic"
]
BROAD_MEDIA_PERSON = [
    "image", "images", "photo", "photos", "picture", "pictures",
    "video", "videos", "audio", "voice", "voices", "face", "faces",
    "facial", "person", "people", "identity", "impersonation",
    "impersonate", "clone", "cloned", "fake", "deepfake"
]

def p(*args):
    print(*args, flush=True)

def norm(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def norm_subreddit(x):
    x = norm(x)
    x = re.sub(r"^/?r/", "", x, flags=re.I).strip("/")
    return x or None

def contains_phrase(text, phrase):
    parts = re.split(r"[\s\-]+", phrase.lower().strip())
    pat = r"[\s\-]+".join(re.escape(q) for q in parts if q)
    return bool(re.search(rf"(?<!\w){pat}(?!\w)", text.lower()))

def broad_prefilter(title, selftext):
    text = f"{norm(title)} {norm(selftext)}"
    ai_hits = [x for x in BROAD_AI if contains_phrase(text, x)]
    media_hits = [x for x in BROAD_MEDIA_PERSON if contains_phrase(text, x)]
    explicit = any(
        x in media_hits
        for x in ["deepfake", "fake", "clone", "cloned", "impersonation", "impersonate"]
    )
    return explicit or (bool(ai_hits) and bool(media_hits)), ai_hits, media_hits

def load_candidates(path):
    p("Loading screening workbook ...")
    df = pd.read_excel(path, sheet_name="Community Screening")
    df = df[["Candidate ID", "Subreddit", "Discovery mentions"]].copy()
    df["subreddit_api"] = df["Subreddit"].map(norm_subreddit)
    df = df.dropna(subset=["subreddit_api"]).drop_duplicates("subreddit_api")
    p(f"Loaded {len(df)} candidate communities.")
    return df

def stratified_pilot_sample(df, n=30):
    x = df.sort_values(["Discovery mentions", "Subreddit"], ascending=[False, True]).reset_index(drop=True)
    if len(x) <= n:
        return x.copy()
    k = n // 3
    high = x.head(k)
    low = x.tail(k)
    mid_start = max(0, len(x)//2 - k//2)
    mid = x.iloc[mid_start:mid_start+k]
    sample = pd.concat([high, mid, low]).drop_duplicates("subreddit_api")
    if len(sample) < n:
        rem = x[~x["subreddit_api"].isin(sample["subreddit_api"])]
        sample = pd.concat([sample, rem.head(n-len(sample))])
    return sample.head(n).reset_index(drop=True)

def month_windows():
    cur = pd.Timestamp(START)
    end = pd.Timestamp(BEFORE)
    while cur < end:
        nxt = min(cur + pd.offsets.MonthBegin(1), end)
        if nxt <= cur:
            nxt = min(cur + pd.DateOffset(months=1), end)
        yield cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")
        cur = nxt

def fetch_window(subreddit, after, before):
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "limit": "auto",
        "sort": "asc",
        "fields": "id,subreddit,created_utc,title,selftext,url,score,num_comments"
    }
    last_error = None
    last_url = None
    for attempt in range(2):
        try:
            r = SESSION.get(API, params=params, timeout=TIMEOUT)
            last_url = r.url
            if r.status_code == 429:
                time.sleep(3)
                continue
            r.raise_for_status()
            payload = r.json()
            data = payload.get("data", payload if isinstance(payload, list) else [])
            return data if isinstance(data, list) else [], None, r.url
        except Exception as e:
            last_error = repr(e)
            time.sleep(1.0 * (attempt + 1))
    return [], last_error, last_url

def validate_known_positive():
    p("Running known-positive check ...")
    posts, err, _ = fetch_window("antiai", "2026-07-21", "2026-07-23")
    if err:
        p("Known-positive API error:", err)
        return False
    target = next((x for x in posts if str(x.get("id")) == KNOWN_ID), None)
    if not target:
        p("Known-positive post not returned.")
        return False
    keep, ai, media = broad_prefilter(target.get("title"), target.get("selftext"))
    p("=== KNOWN-POSITIVE CHECK ===")
    p("Returned by Arctic Shift: YES")
    p("Sensitive prefilter keeps it:", "YES" if keep else "NO")
    p("AI/context hits:", " | ".join(ai))
    p("Media/person hits:", " | ".join(media))
    p("RESULT:", "PASS" if keep else "FAIL")
    return bool(keep)

def append_csv(df, path):
    if df is None or df.empty:
        return
    df.to_csv(path, mode="a", header=not path.exists(), index=False)

def run_pilot(screening_xlsx, pilot_n=30, resume=True):
    p("Starting pilot ...")
    if not validate_known_positive():
        raise RuntimeError("STOP: known-positive validation failed.")

    candidates = load_candidates(screening_xlsx)
    pilot = stratified_pilot_sample(candidates, pilot_n)

    done = set()
    if resume and SUMMARY_CKPT.exists():
        old = pd.read_csv(SUMMARY_CKPT)
        done = set(old["Candidate ID"].astype(str))
        p(f"Resume mode: {len(done)} communities already completed.")

    todo = pilot[~pilot["Candidate ID"].astype(str).isin(done)].copy()
    p(f"Pilot communities total: {len(pilot)}")
    p(f"Still to process: {len(todo)}")
    p("Checkpointing after every subreddit.\n")

    windows = list(month_windows())

    for idx, (_, c) in enumerate(todo.iterrows(), start=1):
        sub = c["subreddit_api"]
        p(f"[{idx}/{len(todo)}] r/{sub} — starting")

        unique_all_ids = set()
        kept_ids = set()
        community_candidates = []
        community_audit = []
        failures = 0
        caps = 0
        successful = 0

        for widx, (a, b) in enumerate(windows, start=1):
            posts, err, url = fetch_window(sub, a, b)

            if err:
                failures += 1
                community_audit.append({
                    "Candidate ID": c["Candidate ID"], "Subreddit": c["Subreddit"],
                    "After": a, "Before": b, "Returned posts": None,
                    "Potentially capped": None, "Error": err, "Request URL": url
                })
            else:
                successful += 1
                capped = len(posts) >= 1000
                caps += int(capped)
                community_audit.append({
                    "Candidate ID": c["Candidate ID"], "Subreddit": c["Subreddit"],
                    "After": a, "Before": b, "Returned posts": len(posts),
                    "Potentially capped": capped, "Error": "", "Request URL": url
                })

                for post in posts:
                    pid = str(post.get("id", "")).strip()
                    if not pid:
                        continue
                    unique_all_ids.add(pid)
                    keep, ai, media = broad_prefilter(post.get("title"), post.get("selftext"))
                    if not keep or pid in kept_ids:
                        continue
                    kept_ids.add(pid)
                    dt = pd.to_datetime(post.get("created_utc"), unit="s", utc=True, errors="coerce")
                    community_candidates.append({
                        "Candidate ID": c["Candidate ID"], "Subreddit": c["Subreddit"],
                        "post_id": pid, "created_utc": dt, "title": norm(post.get("title")),
                        "selftext": norm(post.get("selftext")), "url": post.get("url"),
                        "score": post.get("score"), "num_comments": post.get("num_comments"),
                        "AI/context hits": " | ".join(ai),
                        "Media/person hits": " | ".join(media),
                        "Retrieval status": "CANDIDATE ONLY – NOT FINAL RELEVANCE"
                    })

            if widx % 12 == 0:
                p(f"    {widx}/{len(windows)} monthly windows processed ...")

            time.sleep(SLEEP)

        cdf = pd.DataFrame(community_candidates)

        if cdf.empty:
            first = last = None
            span = None
            active_months = 0
            active_years = 0
        else:
            dates = pd.to_datetime(cdf["created_utc"], utc=True, errors="coerce")
            first = dates.min()
            last = dates.max()
            span = int((last-first).total_seconds()//86400) if len(cdf) >= 2 else None
            naive = dates.dt.tz_convert("UTC").dt.tz_localize(None)
            active_months = naive.dt.to_period("M").nunique()
            active_years = naive.dt.year.nunique()

        if failures == 0 and caps == 0:
            status = "complete_probe"
        elif successful == 0:
            status = "probe_failed"
        else:
            status = "partial_or_capped_probe"

        denom = len(unique_all_ids)
        srow = pd.DataFrame([{
            "Candidate ID": c["Candidate ID"], "Subreddit": c["Subreddit"],
            "Discovery mentions": c["Discovery mentions"], "Probe status": status,
            "Unique posts returned": denom, "Sensitive candidate posts": len(kept_ids),
            "Candidate share": (len(kept_ids)/denom) if denom else 0.0,
            "First candidate post": first, "Last candidate post": last,
            "Temporal span days": span, "Active candidate months": active_months,
            "Active candidate years": active_years, "Successful monthly windows": successful,
            "Failed monthly windows": failures, "Potentially capped monthly windows": caps,
            "FINAL ELIGIBILITY": "NOT DECIDED – PILOT ONLY"
        }])

        append_csv(srow, SUMMARY_CKPT)
        append_csv(cdf, POSTS_CKPT)
        append_csv(pd.DataFrame(community_audit), AUDIT_CKPT)

        p(
            f"[{idx}/{len(todo)}] r/{sub} — DONE | "
            f"posts={denom} | candidates={len(kept_ids)} | "
            f"failures={failures} | caps={caps}"
        )
        p(f"Checkpoint saved: {SUMMARY_CKPT}\n")

    summary = pd.read_csv(SUMMARY_CKPT)
    posts = pd.read_csv(POSTS_CKPT) if POSTS_CKPT.exists() else pd.DataFrame()
    audit = pd.read_csv(AUDIT_CKPT) if AUDIT_CKPT.exists() else pd.DataFrame()

    clean = summary[summary["Probe status"].eq("complete_probe")].copy()
    diagnostics = pd.DataFrame([
        {"Metric":"Pilot communities","Value":len(summary)},
        {"Metric":"Technically complete probes","Value":len(clean)},
        {"Metric":"Communities with >=1 sensitive candidate","Value":int(clean["Sensitive candidate posts"].ge(1).sum())},
        {"Metric":"Communities with >=2 sensitive candidates","Value":int(clean["Sensitive candidate posts"].ge(2).sum())},
        {"Metric":"Communities with >=5 sensitive candidates","Value":int(clean["Sensitive candidate posts"].ge(5).sum())},
        {"Metric":"Communities with >=10 sensitive candidates","Value":int(clean["Sensitive candidate posts"].ge(10).sum())}
    ])

    out = OUTDIR / "Bounded_Sensitive_Retrieval_Pilot_30_RESULTS.xlsx"
    with pd.ExcelWriter(out, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Community Summary", index=False)
        summary.sort_values(["Sensitive candidate posts","Subreddit"], ascending=[False,True]).to_excel(writer, sheet_name="Distribution", index=False)
        diagnostics.to_excel(writer, sheet_name="Pilot Diagnostics", index=False)
        posts.to_excel(writer, sheet_name="Candidate Posts", index=False)
        audit.to_excel(writer, sheet_name="Monthly API Audit", index=False)
        pilot.to_excel(writer, sheet_name="Pilot Sample", index=False)

    p("\n=== PILOT DIAGNOSTICS ===")
    p(diagnostics.to_string(index=False))
    p("\nSaved:", out)

    return {"summary":summary, "posts":posts, "audit":audit, "diagnostics":diagnostics, "output":str(out)}

if __name__ == "__main__":
    import sys
    from pathlib import Path

    print("✅ Script started.", flush=True)

    screening_file = Path("Subreddit_Community_Screening_CURRENT.xlsx")

    if not screening_file.exists():
        print("❌ Missing file: Subreddit_Community_Screening_CURRENT.xlsx", flush=True)
        print("Upload the Excel file into the same Colab runtime and run the script again.", flush=True)
        sys.exit(1)

    try:
        import openpyxl  # noqa: F401
    except ImportError:
        print("❌ Missing package: openpyxl", flush=True)
        print("Run this once in Colab: !pip -q install openpyxl", flush=True)
        sys.exit(1)

    try:
        result = run_pilot(
            "Subreddit_Community_Screening_CURRENT.xlsx",
            pilot_n=30,
            resume=True
        )
        print("✅ PILOT FINISHED.", flush=True)
        print("Excel:", result["output"], flush=True)
    except Exception as e:
        print("❌ ERROR:", repr(e), flush=True)
        raise
